# Passing Network Analysis

This notebook demonstrates how to use `pass_network.py` and how to interpret the resulting centrality metrics. It assumes you have already run the main pipeline to produce `output_videos/events.csv`.



## 1. Inspect raw pass events


## 2. Build the passing network artifacts

Run the CLI once; since this notebook uses highlight clips rather than a full match, we disable period slicing and keep a small edge set.


In [15]:
!python pass_network.py --events_path output_videos/events.csv --output_dir pass_network_outputs --period_strategy none --min_edge_count 3 --max_edges_to_plot 25 --node_label_top_k 5 --meters_per_pixel 0.0 --min_pass_travel 2.0


[pass_network] Top passers (by completed count):
[pass_network]   Player 9: 6 passes [OK]
[pass_network]   Player 21: 6 passes [OK]
[pass_network]   Player 20: 4 passes [LOW]
[pass_network] Saved network plot: pass_network_outputs\plots\pass_network_team1_full_match.png
[pass_network] Top passers (by completed count):
[pass_network]   Player 6: 14 passes [OK]
[pass_network]   Player 1: 12 passes [OK]
[pass_network]   Player 139: 12 passes [OK]
[pass_network]   Player 133: 12 passes [OK]
[pass_network]   Player 11: 11 passes [OK]
[pass_network] Saved network plot: pass_network_outputs\plots\pass_network_team2_full_match.png
[pass_network] Wrote report: pass_network_outputs\pass_network_report.json
[pass_network] Wrote edge list: pass_network_outputs\pass_network_edges.csv


## 3. Player pass summary


In [16]:
import glob
from IPython.display import display

summary_files = sorted(glob.glob("pass_network_outputs/player_summary_*.csv"))
print(f"Found {len(summary_files)} summary files. Showing the first one:")
if summary_files:
    summary_df = pd.read_csv(summary_files[0])
    display(summary_df[['player_id','passes_attempted','passes_completed','pass_completion_rate']].round(3))
else:
    print("No summary files found.")


Found 2 summary files. Showing the first one:


,player_id,passes_attempted,passes_completed,pass_completion_rate
0,21,6.0,0.0,0.0
1,1,0.0,0.0,0.0
2,20,4.0,4.0,1.0
3,9,6.0,0.0,0.0
4,5,0.0,0.0,0.0
5,11,0.0,0.0,0.0


## 4. Edge table (top links)

Review the condensed edge list to see which player pairs exchange the most passes.


In [17]:
import pandas as pd
import json
from pathlib import Path

EVENTS_PATH = Path("output_videos/events.csv")
REPORT_PATH = Path("pass_network_outputs/pass_network_report.json")
EDGES_PATH = Path("pass_network_outputs/pass_network_edges.csv")

if not EVENTS_PATH.exists():
    raise FileNotFoundError("Please run main.py first to generate output_videos/events.csv")

passes = pd.read_csv(EVENTS_PATH)
passes.head()


,type,frame,from_id,to_id,team,interception,ball_speed,travel,pos_x,pos_y
0,pass,6,20,21.0,1,False,17.422776,17.422776,611.262197,440.627014
1,pass,7,20,21.0,1,False,20.994620,20.994620,614.083649,442.779144
2,pass,9,20,21.0,1,False,19.088411,19.088411,617.652130,446.950104
3,pass,10,20,21.0,1,False,18.913842,18.913842,619.653603,449.002327
4,pass,11,21,1.0,1,True,18.244630,18.244630,621.655075,451.054550


## 5. Evaluation checklist

- **Centrality vs pass counts** – compare `pagerank` / `betweenness` with raw attempts to spot connectors vs volume distributors.
- **Top hubs** – slice `player_summary_*` files by name to see how rankings change across periods/windows.
- **Threshold tuning** – adjust `--min_edge_count`, `--max_edges_to_plot`, and `--min_pass_travel` if your snippet is short. For coaches, a high betweenness player is often the pivot, whereas a high out-degree may indicate a playmaker spraying the ball around.



In [18]:
!python pass_network.py --events_path output_videos/events.csv --output_dir pass_network_outputs --plot_format png --min_edge_count 2 --meters_per_pixel 0.0


[pass_network] Top passers (by completed count):
[pass_network]   Player 9: 6 passes [OK]
[pass_network]   Player 21: 6 passes [OK]
[pass_network]   Player 20: 4 passes [LOW]
[pass_network] Saved network plot: pass_network_outputs\plots\pass_network_team1_full_match.png
[pass_network] Top passers (by completed count):
[pass_network]   Player 6: 21 passes [OK]
[pass_network]   Player 4: 19 passes [OK]
[pass_network]   Player 1: 12 passes [OK]
[pass_network]   Player 139: 12 passes [OK]
[pass_network]   Player 133: 12 passes [OK]


Traceback (most recent call last):
  File "c:\Users\praba\OneDrive\Desktop\AI_Football\football_analysis\pass_network.py", line 491, in <module>
    main()
  File "c:\Users\praba\OneDrive\Desktop\AI_Football\football_analysis\pass_network.py", line 474, in main
    plot_clean_network(G, counts_df, team_label, period.label, plot_dir,
  File "c:\Users\praba\OneDrive\Desktop\AI_Football\football_analysis\pass_network.py", line 344, in plot_clean_network
    nx.draw_networkx_edges(graph, layout, edgelist=[(u, v)], width=edge_widths[idx],
  File "c:\Users\praba\anaconda3\envs\ai_football\lib\site-packages\networkx\drawing\nx_pylab.py", line 977, in draw_networkx_edges
    edge_pos = np.asarray([(pos[e[0]], pos[e[1]]) for e in edgelist])
  File "c:\Users\praba\anaconda3\envs\ai_football\lib\site-packages\networkx\drawing\nx_pylab.py", line 977, in <listcomp>
    edge_pos = np.asarray([(pos[e[0]], pos[e[1]]) for e in edgelist])
KeyError: 118


In [13]:
if not REPORT_PATH.exists():
    raise FileNotFoundError("pass_network_report.json not found. Run the previous cell to generate it.")

with open(REPORT_PATH, "r", encoding="utf-8") as f:
    report = json.load(f)

team_reports = {f"team_{entry['team']}_{entry['period']}": entry for entry in report}
list(team_reports.keys())[:5]


['team_1_full_match', 'team_2_full_match']

In [14]:
if not EDGES_PATH.exists():
    raise FileNotFoundError("pass_network_edges.csv not found. Run the previous cell to generate it.")

edges = pd.read_csv(EDGES_PATH)
edges.sort_values("count", ascending=False).head(10)


,from_id,to_id,count,avg_length,completion_rate,weight,team,period
4,9,14.0,9,48.597785,1.000000,9,2,full_match
5,133,4.0,7,56.555946,1.000000,7,2,full_match
6,139,8.0,7,61.596675,1.000000,7,2,full_match
0,21,1.0,6,14.886772,0.000000,6,1,full_match
9,14,9.0,6,34.333074,0.166667,6,2,full_match
13,178,6.0,6,50.867442,1.000000,6,2,full_match
12,3,13.0,6,49.240310,1.000000,6,2,full_match
11,1,9.0,6,45.191265,1.000000,6,2,full_match
10,1,3.0,6,77.728170,1.000000,6,2,full_match
14,212,8.0,6,185.701449,1.000000,6,2,full_match


## Evaluation checklist

1. **Centrality vs pass counts** – compare `pagerank` / `betweenness` with raw `count` to identify playmakers who do more than simple distribution.
2. **Top hubs per period** – slice `team_reports` by `period` (first_half, second_half, window_*) and compare the top-3 nodes.
3. **Recommended thresholds** – experiment with `--min_edge_count` to filter out fleeting connections (start at 3–5). Consider adding a future xT (expected threat) weight per edge and a pass-direction heatmap to capture switches of play.
